# Tools with MCP ⏰

The Model Context Protocol (MCP) provides a standardized way to connect AI agents to external tools and data sources. Let's connect to an MCP server using `langchain-mcp-adapters`.

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

OPENAI_API_KEY=<not set>
LANGSMITH_API_KEY=****954e
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient
import nest_asyncio
import sys
import os
from contextlib import asynccontextmanager
import mcp.client.stdio
import langchain_mcp_adapters.sessions

nest_asyncio.apply()

# MONKEYPATCH TO FIX WINDOWS/JUPYTER 'FILENO' ERROR
# The default 'stdio_client' captures sys.stderr at import time, which causes an error
# in Jupyter on Windows because sys.stderr lacks a fileno().
# We patch the function in BOTH the mcp library and the langchain adapter that imports it
# to force using os.devnull for stderr instead.

# 1. Save original function
_original_stdio_client = mcp.client.stdio.stdio_client

# 2. Define wrapper that forces clean stderr
@asynccontextmanager
async def patched_stdio_client(server, errlog=None):
    # We explicitly ignore the passed errlog (which defaults to broken sys.stderr) 
    # and use a fresh handle to os.devnull
    with open(os.devnull, 'w') as devnull:
        async with _original_stdio_client(server, errlog=devnull) as streams:
            yield streams

# 3. Apply patch to both locations
mcp.client.stdio.stdio_client = patched_stdio_client
langchain_mcp_adapters.sessions.stdio_client = patched_stdio_client
print("Applied Windows MCP stdio patch to mcp and langchain modules.")


# Connect to the mcp-time server for timezone-aware operations
# This Go-based server provides tools for current time, relative time parsing,
# timezone conversion, duration arithmetic, and time comparison
mcp_client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "npx",
            "args": ["-y", "@theo.foobar/mcp-time"],
        }
    },
)

# Load tools from the MCP server
mcp_tools = await mcp_client.get_tools()

print(f"Loaded {len(mcp_tools)} MCP tools: {[t.name for t in mcp_tools]}")

Applied Windows MCP stdio patch to mcp and langchain modules.


  + Exception Group Traceback (most recent call last):
  |   File "f:\Cursos_code\langchain_course\.venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3699, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "C:\Users\Jelb7\AppData\Local\Temp\ipykernel_9580\3421015181.py", line 49, in <module>
  |     mcp_tools = await mcp_client.get_tools()
  |                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "f:\Cursos_code\langchain_course\.venv\Lib\site-packages\langchain_mcp_adapters\client.py", line 197, in get_tools
  |     tools_list = await asyncio.gather(*load_mcp_tool_tasks)
  |                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.2544.0_x64__qbz5n2kfra8p0\Lib\asyncio\tasks.py", line 304, in __step_run_and_handle_result
  |     result = coro.send(None)
  |   File "f:\Cursos_code\langchain_course\.venv\Lib\site-packages\langchain_mcp_adapters\tools.p

Create an agent with the MCP-provided time tools.

In [3]:
from langchain.agents import create_agent

agent_with_mcp = create_agent(
    model="amazon.nova-lite-v1:0",
    tools=mcp_tools,
    system_prompt="You are a helpful assistant",
)

Ask about the current time in San Francisco.

In [4]:
result = await agent_with_mcp.ainvoke(
    {"messages": [{"role": "user", "content": "What's the time in SF right now?"}]}
)
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What's the time in SF right now?
================================== Ai Message ==================================
Tool Calls:
  current_time (call_yv1gmPLMpYFyDOlzfkW2hEVO)
 Call ID: call_yv1gmPLMpYFyDOlzfkW2hEVO
  Args:
    format: 2006-01-02 15:04:05 MST
    timezone: America/Los_Angeles
================================= Tool Message =================================
Name: current_time

2025-10-14 08:48:37 PDT
================================== Ai Message ==================================

It's 8:48:37 AM PDT in San Francisco right now (2025-10-14, America/Los_Angeles).
